<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%203/Aula_3_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 3.1 — Por que times? Construindo o time mínimo

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 3 — Times de IA: como fazer agentes trabalharem juntos

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 2** turbinamos o Treinador com tools — Tavily, Wikipedia, política de fontes.

Esta aula é sobre **dividir o trabalho**:

1. Entender a necessidade: o Treinador sozinho numa pergunta de comissão técnica completa
2. Conceito de time em poucas linhas
3. Criar o **Olheiro** (extrai as tools de busca do Treinador)
4. Montar o **time mínimo**: Treinador como líder + Olheiro como especialista
5. Demo do time respondendo a mesma pergunta — comparação direta

Resultado final: **time funcional com 2 agentes**.


## Setup


In [ ]:
!pip install -U agno openai tavily-python wikipedia

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 1 — Entendendo a necessidade

Pergunta para o Treinador:

> *"Faça uma análise pré-jogo do próximo confronto da Seleção: quem está em forma, quem é o adversário, qual seria a estratégia tática?"*

Três aspectos da pergunta:
- levantamento de dados (forma + adversário),
- análise tática (estratégia)
- síntese (resposta organizada).


In [15]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

modelo_treinador = OpenAIChat(id="gpt-5.4-mini")

treinador_solo = Agent(
    name="Treinador",
    description="Assistente do ScoutAI FC sobre a Seleção Brasileira masculina de futebol.",
    model= modelo_treinador,
    instructions=[
        "Você é o Treinador, assistente do ScoutAI FC dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",

        "POLÍTICA DE FONTES — siga rigorosamente:",
        "• Para EVENTOS RECENTES: use Tavily (busca web).",
        "• Para FATOS HISTÓRICOS CONSOLIDADOS: use Wikipedia.",
        "• Para PERGUNTAS CONCEITUAIS: responda direto sem tool.",
        "• Quando a pergunta tiver MÚLTIPLAS NECESSIDADES, combine fontes.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    markdown=True,
)

treinador_solo.print_response(
    "Faz uma análise pré-jogo do próximo confronto da Seleção: "
    "quem está em forma, quem é o adversário, qual seria a estratégia tática?",
    stream=True,
)

Output()

INFO Searching wikipedia for: Brazil national football team 2024 2025 squad results Copa America World Cup         
     qualifiers

ERROR    Error searching Wikipedia for 'Brazil national football team 2024 2025 squad results Copa America World   
         Cup qualifiers': Expecting value: line 1 column 1 (char 0)

### O que dá pra observar:

1. **Os papéis estão misturados.**
2. **Difícil melhorar uma parte sem afetar as outras.**
3. **Difícil escalar.**


---

## Passo 2 — O que é um time de agentes?

```
           Pergunta do usuário
                   │
                   ▼
            Treinador (líder)
                   │
        ┌──────────┴──────────┐
        ▼                     ▼
    Olheiro              (na Aula 3.2:
   (busca dados)          Analista,
                          que interpreta)
        │                     │
        └──────────┬──────────┘
                   ▼
            Treinador integra
                   │
                   ▼
              Resposta final
```




---

## Passo 3 — Criando o Olheiro




In [16]:
modelo_olheiro = OpenAIChat(id="gpt-5.4-mini")

olheiro = Agent(
    name="Olheiro",
    role="Busca informação verificável sobre futebol em fontes externas (web e Wikipedia)",
    model=modelo_olheiro,
    instructions=[
        "Você é o Olheiro do ScoutAI FC. Sua função é buscar informação verificável em fontes externas.",
        "Use Tavily para eventos recentes (últimos jogos, convocações, forma de jogadores).",
        "Use Wikipedia para fatos históricos consolidados (Copas, biografias, técnicos antigos).",
        "Retorne dados objetivos — números, datas, nomes, fatos. Não interprete nem opine.",
        "Se a busca falhar, diga claramente em vez de chutar.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    markdown=True,
)

---

## Passo 4 — Montando o time mínimo

Agora vem o conceito central da aula: **a classe `Team`** do Agno.


In [17]:
from agno.team import Team

modelo_treinador = OpenAIChat(id="gpt-5.4-mini")

time_scoutai = Team(
    name="Treinador do ScoutAI FC",
    mode="coordinate",
    members=[olheiro],
    model=modelo_treinador,
    instructions=[
        "Você é o Treinador do ScoutAI FC, líder da comissão técnica.",
        "Sua função é responder ao usuário sobre a Seleção Brasileira masculina, "
        "delegando ao Olheiro quando precisar de dados externos verificáveis.",
        "Quando o usuário pedir algo que envolva dados recentes, números ou fatos verificáveis, "
        "delegue ao Olheiro antes de responder.",
        "Quando o usuário pedir conceito tático, explicação histórica conhecida ou opinião analítica, "
        "responda direto sem delegar.",
        "Sempre integre os dados do Olheiro com sua análise tática antes de responder ao usuário.",
        "Responda em português do Brasil, com tom profissional e analítico.",
    ],
    markdown=True,
)

### Anatomia do que acabamos de fazer

| Conceito do Agno | Equivalente no futebol |
|---|---|
| `Team` | A comissão técnica como um todo |
| `members` | Os especialistas (Olheiro, e em breve Analista) |
| `mode="coordinate"` | O Treinador decide quem chamar e quando |
| `instructions` do Team | A "filosofia de trabalho" que o Treinador segue como líder |
| `role` de cada membro | O cargo formal de cada especialista |



---

## Passo 5 — Fluxo do time em ação

1. O Treinador-líder **identifica** que a pergunta tem dados verificáveis (forma, adversário)
2. **Delega ao Olheiro** essa parte
3. **Recebe os dados** e faz a análise tática (estratégia)
4. **Integra** tudo numa resposta final

Observe: `Running: transfer_task_to_member(...)`


In [18]:
time_scoutai.print_response(
    "Faz uma análise pré-jogo do próximo confronto da Seleção: "
    "quem está em forma, quem é o adversário, qual seria a estratégia tática?",
    stream=True,
)

Output()

INFO Searching wikipedia for: Brazil national football team 2026 FIFA World Cup qualification fixtures

### Comparando: Treinador solo vs time

| Aspecto | Treinador solo (Passo 1) | Time (Passo 5) |
|---|---|---|
| **Quem busca os dados** | O próprio Treinador, intercalando com análise | O Olheiro, em chamada delegada explícita |
| **Onde está a análise tática** | Misturada com os dados ao longo da resposta | Concentrada no Treinador, depois de receber dados do Olheiro |
| **Como melhorar a busca** | Mexer nas instructions do Treinador (e arriscar quebrar a parte tática) | Mexer só nas instructions do Olheiro |



### Estado atual do produto

```
ScoutAI FC
│
├── ✅ Time mínimo funcionando
│   ├── Treinador (líder)
│   └── Olheiro (busca: Tavily + Wikipedia)
│
├── ❌ Sem especialista em interpretação tática   → Aula 3.2 (Analista)
├── ❌ Sem memória de sessão                      → Aula 3.2
├── ❌ Sem acesso a base interna (RAG)            → Aula 3.3
└── ❌ Sem fluxo determinístico                   → Aula 4 (Workflows)
```

### Próxima aula

**Aula 3.2 — Adicionando o Analista (com tools de visualização)**

